# vLLM
LLM 모델을 가속화 시킨 버전. 파인튜닝은 못함. 대신 답변 속도가 매우 빠릅니다. 리눅스에서밖에 사용이 안된다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "allganize/Llama-3-Alpha-Ko-8B-Evo"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/6.11G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
%%time
device = "cuda:0"

model.to(device)

messages = [
    {"role": "system", "content": "당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요."},
    {"role": "user", "content": "은행의 기준 금리에 대해서 설명해줘"}
]


encodeds = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
model_inputs = encodeds.to(device)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

generated_ids = model.generate(model_inputs, max_new_tokens=512, eos_token_id=terminators, do_sample=True, repetition_penalty=1.05,)
decoded = tokenizer.batch_decode(generated_ids)
print(decoded[0])

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요.<|eot_id|><|start_header_id|>user<|end_header_id|>

은행의 기준 금리에 대해서 설명해줘<|eot_id|><|start_header_id|>assistant<|end_header_id|>

다행입니다! 은행 기준금리란 은행이 대출자에게 대출금리를 정하는데 사용하는 기준으로서 여신 금리라고도 합니다. 은행 기준금리는 연 이율로 표시되며, 예금의 이자율이나 대출의 이자율과 같이 지급해야 할 수익률입니다.

은행 기준금리는 연방의료당국의 기준 금리와는 별개로 은행 자체적으로 매달 정하지만, 정기적인 정책 평가를 통해 정해지게 됩니다. 은행 기준금리가 낮으면 대출자들은 보다 낮은 금리로 대출을 받을 수 있습니다. 즉, 낮은 금리에 따라 대출금리가 내려간다고 보면 됩니다.

예를 들면, 기준금리가 3퍼센트면 대출자들은 한 달간 3900원씩 대출을 연체할 경우에는 3퍼센트인 기준금리를 적용받게 됩니다. 따라서 대출금리 계산에서 기준금리가 가중치를 두게 되는데, 이는 대출자의 이자 부담을 감소시키도록 도움을 주기도 합니다.

일반인에게 가지는 영향은 다음과 같습니다. 

▶ 대출이자 감소

 은행 기준금리의 하락은 대출자의 이자 부담을 덜어주며 지속적이 저금리 시대를 만들 수 있습니다.

▶ 예금 이자 감소

 연동된 예금의 이자가 기준금리 수준으로 줄어들 것입니다.

▶ 경제성장 및 유동성 확보
이 때 대출금리가 낮으므로 대출이 활발해지고 이른 바에 기업 투자가 활발해야 기업성장에 기여하기 때문입니다.

최근 대출금리는 지속적이 하락하고 있으며, 이는 경제성장 등 비즈니스 환경적 요인에 영향을 받습니다.

또한 기준금리는 인플레이션 조정 차원에서도 중요한 요소가 됩니다. 예를 들면, 높다면 높은 인플레이션이 예상될 수 있으므로 그 예방을 위한 대책으로 

In [ ]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB

In [ ]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])? y


In [ ]:
import string
from vllm import LLM, SamplingParams

# LLM
llm = LLM(model="allganize/Llama-3-Alpha-Ko-8B-Evo")

/usr/local/lib/python3.10/dist-packages/vllm/connections.py:8: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from vllm.version import __version__ as VLLM_VERSION
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 10-17 02:36:03 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='allganize/Llama-3-Alpha-Ko-8B-Evo', speculative_config=None, tokenizer='allganize/Llama-3-Alpha-Ko-8B-Evo', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=allganize/Llama-3-Alpha-Ko-8B-Evo, use_v2_block_manager=True, num_scheduler_steps=1, chunked_prefill_enabled

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 10-17 02:36:11 model_runner.py:1071] Loading model weights took 14.9595 GB
INFO 10-17 02:36:12 gpu_executor.py:122] # GPU blocks: 9689, # CPU blocks: 2048
INFO 10-17 02:36:12 gpu_executor.py:126] Maximum concurrency for 8192 tokens per request: 18.92x
INFO 10-17 02:36:14 model_runner.py:1402] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 10-17 02:36:14 model_runner.py:1406] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 10-17 02:36:41 model_runner.py:1530] Graph capturing finished in 27 secs.


In [ ]:
%%time

# 프롬프트 템플릿 준비
template = string.Template("""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

${system_content}<|eot_id|><|start_header_id|>user<|end_header_id|>

${user_content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
""")

# 메시지 준비
messages = [
    {"role": "system", "content": "당신은 인공지능 어시스턴트입니다. 묻는 말에 친절하고 정확하게 답변하세요."},
    {"role": "user", "content": "은행의 기준 금리에 대해서 설명해줘"}
]

# 프롬프트 생성
prompt = template.safe_substitute({
    "system_content": messages[0]["content"],
    "user_content": messages[1]["content"]
})

# 샘플링 파라미터 설정
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=512,
    stop=["\n<|end_of_text|>", "<|eot_id|>"],  # 문자열로 중지 토큰 지정
    repetition_penalty=1.05
)

# 생성
outputs = llm.generate([prompt], sampling_params)

# 결과 출력
for output in outputs:
    print()
    print(output.outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [00:07<00:00,  7.52s/it, est. speed input: 7.05 toks/s, output: 67.97 toks/s]


은행의 기준 금리란 은행이 자금을 조달하는 데 필요한 최소한의 이자율을 말합니다. 은행의 기준 금리는 은행이 예금이나 대출을 하기 위해 사용하는 금리의 기준이 됩니다.

일반적으로 기준 금리를 적용하는 이유는 은행이 예금을 받으면 돈을 빌려주는 차원에서 예금주에게 이자를 지급해야 하므로, 예금의 이자율을 기준으로 하여야 합니다. 또한 은행이 대출을 하면서 대출자의 이자를 책정하기 위해서도 기준 금리가 필요합니다.

은행의 기준 금리는 여러 가지 요인에 의해 결정됩니다. 가장 중요한 요인은 중앙은행이 정하는 기준 금리입니다. 중앙은행이 기준 금리를 낮추면 은행이 자금을 조달하기 쉽게 되고, 대출의 비용도 낮아지게 됩니다. 하지만 기준 금리를 높이면 은행이 자금을 조달하기 어려워지고 대출의 비용도 높아집니다.

또 다른 중요한 요인은 은행이 보유하고 있는 자산의 가치와 위험도입니다. 은행이 보유하는 자산을 평가한 결과가 좋다면 기준 금리를 낮추기도 하고, 그렇지 않다면 기준 금리를 높이기도 합니다.

기준 금리에는 여러 종류가 있습니다. 예를 들어 국채 금리, LIBOR(리스크프리 라이브러리 금리), T-비트 금리가 있습니다. 각 금리의 특징과 영향이 다릅니다.

국채 금리는 정부가 발행한 채권의 수익률이며, 은행이 자금을 조달하는데 필요한 금리로써 상당히 중요합니다. T-비트 금리는 미국 연방준비제도 이사회(FRB)가 발표하는 금리입니다. FRB가 기준 금리를 낮추면 은행이 자금을 조달하기 쉬워지고 대출의 비용도 낮아집니다.

기준 금리의 영향은 매우 커서 은행, 기업, 개인 모두에게 큰 영향을 미칩니다. 기준 금리가 낮아지면 대출 비용이 낮아져 소비자와 기업이 부담이 줄어들지만, 기준 금리가 높아지면 대출 비용이 높아져 소비자와 기업이 부담이 커집니다.
CPU times: user 7.54 s, sys: 32 ms, total: 7.57 s
Wall time: 7.53 s
